See how many areas/parks have been completed

In [414]:
import pandas as pd
import geopandas as gpd
import os

In [415]:
output_dir = "../workflow_outputs/"
filenames_file = "england_filenames.csv"
park_id_file = "all_parks_ids.csv"

In [416]:
filenames = pd.read_csv(filenames_file)
park_ids = pd.read_csv(park_id_file)

In [417]:
# want a subset of the parks that are in process, so we can check the progress of the workflow
# first_park_index = 29
# last_park_index = first_park_index + 30
# filenames= filenames.iloc[first_park_index:last_park_index]

In [418]:
filenames

,filename
0,Hartlepool_pp_or_g_cmb.geojson
1,Middlesbrough_pp_or_g_cmb.geojson
2,Redcar and Cleveland_pp_or_g_cmb.geojson
3,Stockton-on-Tees_pp_or_g_cmb.geojson
4,Darlington_pp_or_g_cmb.geojson
...,...
291,Trafford_pp_or_g_cmb.geojson
292,Wigan_pp_or_g_cmb.geojson
293,Knowsley_pp_or_g_cmb.geojson
294,Liverpool_pp_or_g_cmb.geojson


In [419]:
# for each filename in filenames
# get foldername by removing the .geojson
# check if the folder exists in output_dir
# add existing folder to a list

in_process_areas = []
for filename in filenames["filename"]:
    foldername = filename.replace(".geojson", "")
    folderpath = output_dir + foldername
    if os.path.exists(folderpath):
        in_process_areas.append(foldername)

In [420]:
100*len(in_process_areas)/len(filenames)

19.93243243243243

In [421]:
in_process_area_names = []
for filename in in_process_areas:
    areaname = filename.replace("_pp_or_g_cmb", "")
    in_process_area_names.append(areaname)

In [422]:
area_df = pd.DataFrame({"area_name": in_process_area_names,
                        "filename": in_process_areas,
                        "folder_path": [output_dir + filename for filename in in_process_areas]})

In [423]:
park_ids

,country,region_id,authority_id,auth_name_e,old_park_id,new_park_id
0,Wales,W10000009,W06000011,Swansea,SWANS_841e2e988f55;SWANS_9add957b9b60,3816d9f14391
1,Wales,W10000009,W06000011,Swansea,SWANS_64180ad89ee2;SWANS_7875d4152bbb,eeb8a45abba6
2,Wales,W10000009,W06000011,Swansea,SWANS_0077e0499336;SWANS_003142c8af8d,2afa4425d6cc
3,Wales,W10000009,W06000011,Swansea,SWANS_8c5a5b68c7e1;SWANS_fb7bfbb4dae0,ee93e1ee473b
4,Wales,W10000009,W06000011,Swansea,SWANS_8119996c470b,a7f47d7bf887
...,...,...,...,...,...,...
217614,England,E12000002,E08000013,St. Helens,STHEL_6a530cb0f057,e97074068e56
217615,England,E12000002,E08000013,St. Helens,STHEL_2df5277ef0c9,f9627d5e096b
217616,England,E12000002,E08000013,St. Helens,STHEL_04f0db6f2403,7957ab4ecde8
217617,England,E12000002,E08000013,St. Helens,STHEL_0384f41414f0,42c0619d2802


In [424]:
area_df

,area_name,filename,folder_path
0,Hartlepool,Hartlepool_pp_or_g_cmb,../workflow_outputs/Hartlepool_pp_or_g_cmb
1,Middlesbrough,Middlesbrough_pp_or_g_cmb,../workflow_outputs/Middlesbrough_pp_or_g_cmb
2,Redcar and Cleveland,Redcar and Cleveland_pp_or_g_cmb,../workflow_outputs/Redcar and Cleveland_pp_or...
3,Stockton-on-Tees,Stockton-on-Tees_pp_or_g_cmb,../workflow_outputs/Stockton-on-Tees_pp_or_g_cmb
4,Darlington,Darlington_pp_or_g_cmb,../workflow_outputs/Darlington_pp_or_g_cmb
5,Halton,Halton_pp_or_g_cmb,../workflow_outputs/Halton_pp_or_g_cmb
6,Warrington,Warrington_pp_or_g_cmb,../workflow_outputs/Warrington_pp_or_g_cmb
7,Blackburn with Darwen,Blackburn with Darwen_pp_or_g_cmb,../workflow_outputs/Blackburn with Darwen_pp_o...
8,Blackpool,Blackpool_pp_or_g_cmb,../workflow_outputs/Blackpool_pp_or_g_cmb
9,"Kingston upon Hull, City of","Kingston upon Hull, City of_pp_or_g_cmb","../workflow_outputs/Kingston upon Hull, City o..."


In [425]:
# for each folder_path in the area_df, find all parks in the area
# in the park_ids dataframe, find all parks with area_name in the area_df
# add the total number of parks in the area to the area_df
area_df["num_parks_total"] = area_df["area_name"].apply(lambda x: len(park_ids[park_ids["auth_name_e"] == x]))
area_df["expected_park_files"] = area_df["num_parks_total"].apply(lambda x: x * 2)

In [426]:
# then count the number of parks in the associated folder_path and add that to the area_df
def count_files_in_folder(folder_path):
    park_files = [f for f in os.listdir(folder_path) if f.endswith(".geojson")]
    return len(park_files)

area_df["completed_files"] = area_df["folder_path"].apply(count_files_in_folder)
area_df["completed_parks"] = area_df["completed_files"] / 2

In [427]:
area_df

,area_name,filename,folder_path,num_parks_total,expected_park_files,completed_files,completed_parks
0,Hartlepool,Hartlepool_pp_or_g_cmb,../workflow_outputs/Hartlepool_pp_or_g_cmb,38,76,76,38.0
1,Middlesbrough,Middlesbrough_pp_or_g_cmb,../workflow_outputs/Middlesbrough_pp_or_g_cmb,75,150,150,75.0
2,Redcar and Cleveland,Redcar and Cleveland_pp_or_g_cmb,../workflow_outputs/Redcar and Cleveland_pp_or...,75,150,150,75.0
3,Stockton-on-Tees,Stockton-on-Tees_pp_or_g_cmb,../workflow_outputs/Stockton-on-Tees_pp_or_g_cmb,393,786,786,393.0
4,Darlington,Darlington_pp_or_g_cmb,../workflow_outputs/Darlington_pp_or_g_cmb,65,130,130,65.0
5,Halton,Halton_pp_or_g_cmb,../workflow_outputs/Halton_pp_or_g_cmb,95,190,190,95.0
6,Warrington,Warrington_pp_or_g_cmb,../workflow_outputs/Warrington_pp_or_g_cmb,173,346,346,173.0
7,Blackburn with Darwen,Blackburn with Darwen_pp_or_g_cmb,../workflow_outputs/Blackburn with Darwen_pp_o...,101,202,202,101.0
8,Blackpool,Blackpool_pp_or_g_cmb,../workflow_outputs/Blackpool_pp_or_g_cmb,82,164,164,82.0
9,"Kingston upon Hull, City of","Kingston upon Hull, City of_pp_or_g_cmb","../workflow_outputs/Kingston upon Hull, City o...",155,310,310,155.0


In [428]:
area_df["pct_complete"] = 100* area_df["completed_files"]/area_df["expected_park_files"]

In [429]:
area_df

,area_name,filename,folder_path,num_parks_total,expected_park_files,completed_files,completed_parks,pct_complete
0,Hartlepool,Hartlepool_pp_or_g_cmb,../workflow_outputs/Hartlepool_pp_or_g_cmb,38,76,76,38.0,100.000000
1,Middlesbrough,Middlesbrough_pp_or_g_cmb,../workflow_outputs/Middlesbrough_pp_or_g_cmb,75,150,150,75.0,100.000000
2,Redcar and Cleveland,Redcar and Cleveland_pp_or_g_cmb,../workflow_outputs/Redcar and Cleveland_pp_or...,75,150,150,75.0,100.000000
3,Stockton-on-Tees,Stockton-on-Tees_pp_or_g_cmb,../workflow_outputs/Stockton-on-Tees_pp_or_g_cmb,393,786,786,393.0,100.000000
4,Darlington,Darlington_pp_or_g_cmb,../workflow_outputs/Darlington_pp_or_g_cmb,65,130,130,65.0,100.000000
5,Halton,Halton_pp_or_g_cmb,../workflow_outputs/Halton_pp_or_g_cmb,95,190,190,95.0,100.000000
6,Warrington,Warrington_pp_or_g_cmb,../workflow_outputs/Warrington_pp_or_g_cmb,173,346,346,173.0,100.000000
7,Blackburn with Darwen,Blackburn with Darwen_pp_or_g_cmb,../workflow_outputs/Blackburn with Darwen_pp_o...,101,202,202,101.0,100.000000
8,Blackpool,Blackpool_pp_or_g_cmb,../workflow_outputs/Blackpool_pp_or_g_cmb,82,164,164,82.0,100.000000
9,"Kingston upon Hull, City of","Kingston upon Hull, City of_pp_or_g_cmb","../workflow_outputs/Kingston upon Hull, City o...",155,310,310,155.0,100.000000
